In [ ]:
import sqlite3

# Connect to the database (or create it if it doesn't exist)
conn = sqlite3.connect('microfinance.db')
cursor = conn.cursor()

# Enable foreign key enforcement
cursor.execute("PRAGMA foreign_keys = ON;")

# Create tables using the provided schema
cursor.executescript("""
  -- 1. Branches
  CREATE TABLE IF NOT EXISTS Branches (
      branch_id   INTEGER PRIMARY KEY AUTOINCREMENT,
      branch_name VARCHAR(100) NOT NULL,
      city        VARCHAR(50) NOT NULL,
      region      VARCHAR(50) NOT NULL,
      phone       VARCHAR(20)
  );

  -- 2. Loan_Officers
  CREATE TABLE IF NOT EXISTS Loan_Officers (
      officer_id   INTEGER PRIMARY KEY AUTOINCREMENT,
      full_name    VARCHAR(100) NOT NULL,
      email        VARCHAR(100) UNIQUE NOT NULL,
      hire_date    DATE NOT NULL,
      branch_id    INTEGER NOT NULL,
      FOREIGN KEY (branch_id) REFERENCES Branches(branch_id)
  );

  -- 3. Borrowers
  CREATE TABLE IF NOT EXISTS Borrowers (
      borrower_id  INTEGER PRIMARY KEY AUTOINCREMENT,
      full_name    VARCHAR(100) NOT NULL,
      cnic         VARCHAR(15) UNIQUE NOT NULL,
      phone        VARCHAR(20),
      dob          DATE NOT NULL,
      branch_id    INTEGER NOT NULL,
      FOREIGN KEY (branch_id) REFERENCES Branches(branch_id)
  );

  -- 4. Loan_Products
  CREATE TABLE IF NOT EXISTS Loan_Products (
      product_id      INTEGER PRIMARY KEY AUTOINCREMENT,
      product_name    VARCHAR(100) NOT NULL,
      interest_rate   DECIMAL(5,2) NOT NULL,
      max_amount      DECIMAL(12,2) NOT NULL,
      max_term_months INTEGER NOT NULL
  );

  -- 5. Loans
  CREATE TABLE IF NOT EXISTS Loans (
      loan_id       INTEGER PRIMARY KEY AUTOINCREMENT,
      borrower_id   INTEGER NOT NULL,
      officer_id    INTEGER NOT NULL,
      product_id    INTEGER NOT NULL,
      amount        DECIMAL(12,2) NOT NULL,
      disbursed_on  DATE NOT NULL,
      due_date      DATE NOT NULL,
      status        VARCHAR(20) NOT NULL DEFAULT 'Active',
      FOREIGN KEY (borrower_id) REFERENCES Borrowers(borrower_id),
      FOREIGN KEY (officer_id)  REFERENCES Loan_Officers(officer_id),
      FOREIGN KEY (product_id)  REFERENCES Loan_Products(product_id)
  );

  -- 6. Repayments
  CREATE TABLE IF NOT EXISTS Repayments (
      repayment_id   INTEGER PRIMARY KEY AUTOINCREMENT,
      loan_id        INTEGER NOT NULL,
      amount_paid    DECIMAL(12,2) NOT NULL,
      payment_date   DATE NOT NULL,
      payment_method VARCHAR(30) NOT NULL,
      FOREIGN KEY (loan_id) REFERENCES Loans(loan_id)
  );

  -- 7. Guarantors (junction: Borrowers <-> Loans)
  CREATE TABLE IF NOT EXISTS Guarantors (
      guarantor_id  INTEGER PRIMARY KEY AUTOINCREMENT,
      loan_id       INTEGER NOT NULL,
      borrower_id   INTEGER NOT NULL,
      relationship  VARCHAR(50) NOT NULL,
      FOREIGN KEY (loan_id)     REFERENCES Loans(loan_id),
      FOREIGN KEY (borrower_id) REFERENCES Borrowers(borrower_id)
  );

  -- 8. Collateral
  CREATE TABLE IF NOT EXISTS Collateral (
      collateral_id   INTEGER PRIMARY KEY AUTOINCREMENT,
      loan_id         INTEGER NOT NULL,
      asset_type      VARCHAR(100) NOT NULL,
      estimated_value DECIMAL(12,2) NOT NULL,
      description     TEXT,
      FOREIGN KEY (loan_id) REFERENCES Loans(loan_id)
  );
""")

conn.commit()
print("All tables created successfully.")

In [ ]:
cursor.executescript("""
  -- Branches
  INSERT INTO Branches (branch_name, city, region) VALUES
      ('Lahore Main', 'Lahore', 'Punjab'),
      ('Karachi Central', 'Karachi', 'Sindh'),
      ('Islamabad Blue Area', 'Islamabad', 'Capital'),
      ('Faisalabad East', 'Faisalabad', 'Punjab'),
      ('Peshawar Saddar', 'Peshawar', 'KPK');

  -- Loan_Officers
  INSERT INTO Loan_Officers (full_name, email, hire_date, branch_id) VALUES
      ('Ahmed Raza',    'ahmed.raza@mfbank.pk',    '2020-03-15', 1),
      ('Sara Malik',    'sara.malik@mfbank.pk',     '2019-07-01', 1),
      ('Bilal Khan',    'bilal.khan@mfbank.pk',     '2021-01-10', 2),
      ('Hina Qureshi',  'hina.qureshi@mfbank.pk',   '2018-11-20', 3),
      ('Usman Tariq',   'usman.tariq@mfbank.pk',    '2022-05-05', 4),
      ('Zara Hussain',  'zara.hussain@mfbank.pk',   '2020-09-30', 5),
      ('Kamran Iqbal',  'kamran.iqbal@mfbank.pk',   '2017-04-14', 2),
      ('Nadia Farooq',  'nadia.farooq@mfbank.pk',   '2023-02-28', 3),
      ('Tariq Mehmood', 'tariq.mehmood@mfbank.pk',  '2016-08-19', 4),
      ('Ayesha Siddiq', 'ayesha.siddiq@mfbank.pk',  '2021-12-01', 5);

  -- Borrowers
  INSERT INTO Borrowers (full_name, cnic, phone, dob, branch_id) VALUES
      ('Muhammad Ali',    '35201-1234567-1', '0300-1111111', '1985-04-10', 1),
      ('Fatima Zahra',    '42101-2345678-2', '0301-2222222', '1990-08-22', 2),
      ('Asif Nawaz',      '61101-3456789-3', '0302-3333333', '1978-12-05', 3),
      ('Sana Javed',      '33100-4567890-4', '0303-4444444', '1995-03-17', 4),
      ('Imran Shafi',     '17301-5678901-5', '0304-5555555', '1982-07-30', 5),
      ('Rabia Noor',      '35202-6789012-6', '0305-6666666', '1993-11-11', 1),
      ('Zubair Ahmed',    '42102-7890123-7', '0306-7777777', '1975-01-25', 2),
      ('Mehwish Rana',    '61102-8901234-8', '0307-8888888', '1988-06-14', 3),
      ('Shahid Latif',    '33101-9012345-9', '0308-9999999', '1980-09-03', 4),
      ('Amna Bibi',       '17302-0123456-0', '0309-0000000', '1997-02-28', 5),
      ('Naveed Sultan',   '35203-1122334-1', '0310-1212121', '1983-05-19', 1),
      ('Kiran Bashir',    '42103-2233445-2', '0311-2323232', '1991-10-07', 2);

  -- Loan_Products
  INSERT INTO Loan_Products (product_name, interest_rate, max_amount, max_term_months)
   VALUES
      ('Startup Micro Loan',   12.5,  50000,  12),
      ('Agriculture Loan',     10.0, 200000,  24),
      ('Women Empowerment',     8.0,  75000,  18),
      ('Small Business Loan',  14.0, 500000,  36),
      ('Emergency Loan',       15.0,  25000,   6);

  -- Loans
  INSERT INTO Loans (borrower_id, officer_id, product_id, amount, disbursed_on,
  due_date, status) VALUES
      (1,  1, 1,  45000, '2024-01-15', '2025-01-15', 'Active'),
      (2,  3, 3,  70000, '2024-02-01', '2025-08-01', 'Active'),
      (3,  4, 2, 180000, '2023-11-10', '2025-11-10', 'Active'),
      (4,  5, 4, 450000, '2024-03-20', '2027-03-20', 'Active'),
      (5,  6, 5,  20000, '2024-04-05', '2024-10-05', 'Closed'),
      (6,  2, 1,  50000, '2024-05-12', '2025-05-12', 'Active'),
      (7,  7, 2, 150000, '2023-09-01', '2025-09-01', 'Active'),
      (8,  8, 3,  60000, '2024-06-18', '2025-12-18', 'Active'),
      (9,  9, 4, 300000, '2024-01-30', '2027-01-30', 'Active'),
      (10, 10, 5, 25000, '2024-07-22', '2025-01-22', 'Defaulted'),
      (11,  1, 1,  40000, '2024-08-10', '2025-08-10', 'Active'),
      (12,  3, 3,  75000, '2024-09-05', '2026-03-05', 'Active');

  -- Repayments
  INSERT INTO Repayments (loan_id, amount_paid, payment_date, payment_method) VALUES
      (1,  5000, '2024-02-15', 'Cash'),
      (1,  5000, '2024-03-15', 'Bank Transfer'),
      (2,  8000, '2024-03-01', 'Cash'),
      (3, 15000, '2023-12-10', 'Cheque'),
      (3, 15000, '2024-01-10', 'Bank Transfer'),
      (4, 20000, '2024-04-20', 'Cash'),
      (5, 20000, '2024-07-05', 'Cash'),
      (6,  5000, '2024-06-12', 'Bank Transfer'),
      (7, 12000, '2023-10-01', 'Cheque'),
      (8,  6000, '2024-07-18', 'Cash'),
      (9, 25000, '2024-02-28', 'Bank Transfer'),
      (10, 5000, '2024-08-22', 'Cash');

  -- Guarantors (junction table)
  INSERT INTO Guarantors (loan_id, borrower_id, relationship) VALUES
      (1,  6, 'Sibling'),
      (2,  7, 'Spouse'),
      (3,  8, 'Friend'),
      (4,  9, 'Parent'),
      (5, 10, 'Sibling'),
      (6,  1, 'Spouse'),
      (7,  2, 'Friend'),
      (8,  3, 'Parent'),
      (9,  4, 'Sibling'),
      (10, 5, 'Friend'),
      (11, 12, 'Sibling'),
      (12, 11, 'Spouse');

  -- Collateral
  INSERT INTO Collateral (loan_id, asset_type, estimated_value, description) VALUES
      (1,  'Motorcycle',      35000, 'Honda CD70, 2021 model'),
      (2,  'Livestock',       90000, '3 dairy cows'),
      (3,  'Agricultural Land', 500000, '2 acres in Faisalabad'),
      (4,  'Commercial Property', 1200000, 'Shop in Islamabad'),
      (5,  'Jewellery',       18000, 'Gold set approx 10 grams'),
      (6,  'Motorcycle',      40000, 'Yamaha YBR 125, 2022'),
      (7,  'Livestock',      120000, '5 goats and 2 cows'),
      (8,  'Sewing Machine',  15000, 'Industrial sewing machine'),
      (9,  'Residential Plot', 800000, '5 marla plot in Faisalabad'),
      (10, 'Mobile Phone',     8000, 'Samsung Galaxy A series'),
      (11, 'Motorcycle',      38000, 'Honda Pridor, 2020'),
      (12, 'Jewellery',       50000, 'Gold bangles approx 25 grams');
""")

conn.commit()
print("Sample data inserted successfully.")

In [8]:
import sqlite3

# Reconnect to ensure cursor is active
conn = sqlite3.connect('microfinance.db')
cursor = conn.cursor()

tables = ['Branches', 'Loan_Officers', 'Borrowers', 'Loan_Products',                
          'Loans', 'Repayments', 'Guarantors', 'Collateral']                        
                                                                                   
for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"{table}: {count} rows")

# Optional: conn.close()


Branches: 5 rows
Loan_Officers: 10 rows
Borrowers: 12 rows
Loan_Products: 5 rows
Loans: 12 rows
Repayments: 12 rows
Guarantors: 12 rows
Collateral: 12 rows
